In [157]:
from Utils_app import*
from Utils_eda import(analyze_cutoff_impact)

In [164]:
import pandas as pd

In [141]:
df = pd.read_csv('data/ratings.csv')

In [142]:
df = add_datetime_features(df, timestamp_col="timestamp")

print(df[["timestamp", "datetime", "year", "month", "day"]].head())

    timestamp            datetime  year  month  day
0  1260759144 2009-12-14 02:52:24  2009     12   14
1  1260759179 2009-12-14 02:52:59  2009     12   14
2  1260759182 2009-12-14 02:53:02  2009     12   14
3  1260759185 2009-12-14 02:53:05  2009     12   14
4  1260759205 2009-12-14 02:53:25  2009     12   14


In [143]:
df_cleaned = clean_rating_dataset(df, timestamp_cols=["timestamp"], rating_col="rating")

=== Initial dataset statistics ===
Dataset statistics:
  Rows: 100004
  Unique users: 671
  Unique products: 9066
  Avg ratings per user: 149.04
  Avg ratings per product: 11.03
Removed 5 rows containing nulls (excluding timestamp columns).
Removed 0 duplicate user-product ratings.
Removed 2 rows with min (-1.0) or max (99.0) ratings.
=== Final cleaned dataset statistics ===
Dataset statistics:
  Rows: 99997
  Unique users: 671
  Unique products: 9066
  Avg ratings per user: 149.03
  Avg ratings per product: 11.03


In [156]:
filtered_df=analyze_cutoff_impact(df_cleaned, product_min_ratings=7, user_low_n=10)


=== 1. Product rating counts BEFORE filtering ===
count    9066.000000
mean       11.029892
std        24.049299
min         1.000000
25%         1.000000
50%         3.000000
75%         9.000000
max       341.000000
Name: count, dtype: float64

Products with < 7 ratings: 6273

Records BEFORE filtering: 99997
Records AFTER filtering: 86245
Removed: 13752 records (13.75%)

=== 2. User activity (ratings per user) AFTER filtering ===
Total unique users after filtering: 671

=== Lowest 10 activity levels (explained) ===

This table shows how many users belong to the lowest activity groups.
Columns:
- 'ratings_per_user' → How many ratings a user has (1, 2, 3, ...)
- 'num_users' → How many users fall into that activity level

Example interpretation:
If the row says:
    ratings_per_user = 1 → num_users = 250
It means that 250 users created exactly 1 rating in the dataset.
    


ValueError: cannot insert count, already exists

In [150]:
filtered_df.head()

,user_id,product_id,rating,timestamp,datetime,year,month,day
0,1.0,31.0,2.5,1260759144,2009-12-14 02:52:24,2009,12,14
1,1.0,1029.0,3.0,1260759179,2009-12-14 02:52:59,2009,12,14
2,1.0,1061.0,3.0,1260759182,2009-12-14 02:53:02,2009,12,14
3,1.0,1129.0,2.0,1260759185,2009-12-14 02:53:05,2009,12,14
4,1.0,1172.0,4.0,1260759205,2009-12-14 02:53:25,2009,12,14


In [151]:
filtered_df

,user_id,product_id,rating,timestamp,datetime,year,month,day
0,1.0,31.0,2.5,1260759144,2009-12-14 02:52:24,2009,12,14
1,1.0,1029.0,3.0,1260759179,2009-12-14 02:52:59,2009,12,14
2,1.0,1061.0,3.0,1260759182,2009-12-14 02:53:02,2009,12,14
3,1.0,1129.0,2.0,1260759185,2009-12-14 02:53:05,2009,12,14
4,1.0,1172.0,4.0,1260759205,2009-12-14 02:53:25,2009,12,14
...,...,...,...,...,...,...,...,...
99998,671.0,6212.0,2.5,1065149436,2003-10-03 02:50:36,2003,10,3
100000,671.0,6269.0,4.0,1065149201,2003-10-03 02:46:41,2003,10,3
100001,671.0,6365.0,4.0,1070940363,2003-12-09 03:26:03,2003,12,9
100002,671.0,6385.0,2.5,1070979663,2003-12-09 14:21:03,2003,12,9


In [152]:
print(df.dtypes)
print(df.isna().sum())
df = df.dropna(subset=['rating'])
df['rating'] = df['rating'].astype(float)
df = df.dropna(subset=['rating'])

user_id              float64
product_id           float64
rating               float64
timestamp              int64
datetime      datetime64[ns]
year                   Int64
month                  Int64
day                    Int64
dtype: object
user_id       1
product_id    4
rating        0
timestamp     0
datetime      0
year          0
month         0
day           0
dtype: int64


In [154]:
import pandas as pd
import numpy as np
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity

# 1️⃣ Wczytanie danych
df = filtered_df  # kolumny: user_id, product_id, ratings

df = df.groupby(['user_id', 'product_id'], as_index=False)['rating'].mean()

# 2️⃣ Tworzymy macierz użytkownik x produkt
user_item_matrix = df.pivot(index='user_id', columns='product_id', values='rating').fillna(0)

# 3️⃣ SVD
n_components = 20
svd = TruncatedSVD(n_components=n_components, random_state=42)
svd_latent = svd.fit_transform(user_item_matrix)
approx_matrix = svd_latent @ svd.components_

def svd_recommend(user_id, top_n=20):
    user_idx = user_item_matrix.index.get_loc(user_id)
    user_scores = approx_matrix[user_idx].copy()
    already_rated = user_item_matrix.iloc[user_idx] > 0
    user_scores[already_rated] = -np.inf
    top_items = user_item_matrix.columns[np.argsort(user_scores)[-top_n:][::-1]]
    return top_items.tolist()

# 4️⃣ Item-Based CF
item_similarity = cosine_similarity(user_item_matrix.T)
item_similarity_df = pd.DataFrame(item_similarity, index=user_item_matrix.columns, columns=user_item_matrix.columns)

def item_based_recommend(user_id, top_n=20):
    user_ratings = user_item_matrix.loc[user_id]
    scores = user_ratings.values @ item_similarity
    scores = pd.Series(scores, index=user_item_matrix.columns)
    scores = scores[user_ratings == 0]
    return scores.sort_values(ascending=False).head(top_n).index.tolist()

# 5️⃣ RMSE i MAE
mask = user_item_matrix.values > 0

rmse_svd = np.sqrt(np.mean((approx_matrix[mask] - user_item_matrix.values[mask])**2))
mae_svd = np.mean(np.abs(approx_matrix[mask] - user_item_matrix.values[mask]))

ibcf_pred_matrix = user_item_matrix.values @ item_similarity
rmse_ibcf = np.sqrt(np.mean((ibcf_pred_matrix[mask] - user_item_matrix.values[mask])**2))
mae_ibcf = np.mean(np.abs(ibcf_pred_matrix[mask] - user_item_matrix.values[mask]))

# 6️⃣ Średnia przewidywana ocena dla produktów (globalny ranking)
svd_avg_pred = approx_matrix.mean(axis=0)
ibcf_avg_pred = ibcf_pred_matrix.mean(axis=0)

# 7️⃣ Wyniki
print(f"SVD RMSE: {rmse_svd:.4f}, MAE: {mae_svd:.4f}")
print(f"Item-Based CF RMSE: {rmse_ibcf:.4f}, MAE: {mae_ibcf:.4f}")



# 10️⃣ Przykładowe rekomendacje
example_user = df['user_id'].iloc[0]
print(f"\nSVD TOP-20 dla użytkownika {example_user}: {svd_recommend(example_user)}")
print(f"Item-Based CF TOP-20 dla użytkownika {example_user}: {item_based_recommend(example_user)}")

SVD RMSE: 2.3566, MAE: 2.0303
Item-Based CF RMSE: 340.3721, MAE: 266.1914

SVD TOP-20 dla użytkownika 1.0: [1374.0, 1282.0, 1214.0, 596.0, 750.0, 1272.0, 1276.0, 1208.0, 2529.0, 594.0, 1079.0, 1090.0, 1275.0, 1204.0, 1222.0, 1965.0, 1136.0, 1262.0, 1221.0, 1080.0]
Item-Based CF TOP-20 dla użytkownika 1.0: [1387.0, 1266.0, 2194.0, 1214.0, 3108.0, 1282.0, 1394.0, 2797.0, 1225.0, 2174.0, 919.0, 2987.0, 1302.0, 2289.0, 2366.0, 1097.0, 1234.0, 1208.0, 1252.0, 1079.0]


In [168]:
item_similarity = item_similarity.reindex(index=train_matrix.columns, columns=train_matrix.columns, fill_value=0)

In [173]:
import os
import pandas as pd
import numpy as np
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
import pickle

# 1️⃣ Wczytanie danych
df = filtered_df

# 2️⃣ Split danych na poziomie użytkownika
train_list = []
test_list = []

unique_users = df['user_id'].unique()
for user in unique_users:
    user_data = df[df['user_id'] == user]
    if len(user_data) < 2:
        continue  # pomijamy użytkowników z <2 ocenami
    train_u, test_u = train_test_split(user_data, test_size=0.2, random_state=42)
    train_list.append(train_u)
    test_list.append(test_u)

train_df = pd.concat(train_list)
test_df = pd.concat(test_list)

# 3️⃣ Tworzymy macierz user x product
train_matrix = train_df.pivot(index='user_id', columns='product_id', values='rating').fillna(0)
test_matrix = test_df.pivot(index='user_id', columns='product_id', values='rating').fillna(0)

# Dopasowanie test_matrix do train_matrix (te same produkty i użytkownicy)
test_matrix = test_matrix.reindex(index=train_matrix.index, columns=train_matrix.columns, fill_value=0)

# 4️⃣ MODELE

# 4a️⃣ SVD
n_components = 20
svd = TruncatedSVD(n_components=n_components, random_state=42)
svd_matrix = svd.fit_transform(train_matrix)
approx_matrix = np.dot(svd_matrix, svd.components_)
approx_matrix = pd.DataFrame(approx_matrix, index=train_matrix.index, columns=train_matrix.columns)

# 4b️⃣ Item-Based CF (similarity na produktach)
item_matrix = train_matrix.T
item_similarity = item_matrix.corr(method='pearson').fillna(0)
# upewniamy się, że macierze są w tej samej kolejności
ibcf_pred_matrix = train_matrix.dot(item_similarity.reindex(index=train_matrix.columns, columns=train_matrix.columns))

# 5️⃣ RMSE i MAE
def compute_rmse_mae(pred_matrix, true_matrix):
    mask = true_matrix > 0
    mse = mean_squared_error(true_matrix[mask], pred_matrix[mask])
    mae = mean_absolute_error(true_matrix[mask], pred_matrix[mask])
    return np.sqrt(mse), mae

# svd_rmse, svd_mae = compute_rmse_mae(approx_matrix, test_matrix)
# ibcf_rmse, ibcf_mae = compute_rmse_mae(ibcf_pred_matrix, test_matrix)

# print(f"SVD RMSE: {svd_rmse:.4f}, MAE: {svd_mae:.4f}")
# print(f"Item-Based CF RMSE: {ibcf_rmse:.4f}, MAE: {ibcf_mae:.4f}")

# # 6️⃣ TOP-5 rekomendacje dla przykładowego użytkownika
# def top_n(pred_matrix, user_index, n=5):
#     top_items = np.argsort(pred_matrix.iloc[user_index])[::-1][:n]
#     return pred_matrix.columns[top_items].tolist()

# example_user = 0
# print("SVD TOP-5:", top_n(approx_matrix, example_user))
# print("Item-Based CF TOP-5:", top_n(ibcf_pred_matrix, example_user))

# # 7️⃣ Wybór najlepszego modelu
# if svd_rmse < ibcf_rmse:
#     best_model = ('svd', svd)
# else:
#     best_model = ('ibcf', item_similarity)

# # 8️⃣ Zapis modelu
# os.makedirs('best_model', exist_ok=True)
# model_path = os.path.join('best_model', 'best_model.pkl')
# with open(model_path, 'wb') as f:
#     pickle.dump(best_model, f)

# print(f"Najlepszy model: {best_model[0]} zapisany w {model_path}")


In [186]:
example_user=100

In [189]:
import os
import pandas as pd
import numpy as np
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
import pickle

# 1️⃣ Wczytanie danych
df = filtered_df  # Twój DataFrame z kolumnami: user_id, product_id, rating

# 2️⃣ Split danych na poziomie użytkownika
train_list, test_list = [], []
min_ratings_per_user = 2

for user, user_data in df.groupby('user_id'):
    if len(user_data) < min_ratings_per_user:
        train_list.append(user_data)
        continue
    train_u, test_u = train_test_split(user_data, test_size=0.2, random_state=42)
    train_list.append(train_u)
    test_list.append(test_u)

train_df = pd.concat(train_list)
test_df = pd.concat(test_list) if test_list else pd.DataFrame(columns=df.columns)

# 3️⃣ Macierze user x product
train_matrix = train_df.pivot(index='user_id', columns='product_id', values='rating').fillna(0)
test_matrix = test_df.pivot(index='user_id', columns='product_id', values='rating')
test_matrix = test_matrix.reindex(index=train_matrix.index, columns=train_matrix.columns, fill_value=0)

# 4️⃣ SVD
n_components = 20
svd = TruncatedSVD(n_components=n_components, random_state=42)
svd_matrix = svd.fit_transform(train_matrix)
approx_matrix = np.dot(svd_matrix, svd.components_)
approx_matrix = pd.DataFrame(approx_matrix, index=train_matrix.index, columns=train_matrix.columns)

# 5️⃣ Item-Based CF
item_matrix = train_matrix.T
item_similarity = item_matrix.corr(method='pearson').fillna(0)

# Dopasowanie item_similarity do wspólnych kolumn train_matrix
item_similarity = item_similarity.reindex(index=train_matrix.columns, columns=train_matrix.columns, fill_value=0)

# Predykcje IBCF
ibcf_pred_matrix = train_matrix.dot(item_similarity)
ibcf_pred_matrix = ibcf_pred_matrix.reindex(index=train_matrix.index, columns=train_matrix.columns, fill_value=0)

# 6️⃣ Funkcja do RMSE i MAE
def compute_rmse_mae(pred_matrix, true_matrix):
    pred_matrix = pred_matrix.reindex(index=true_matrix.index, columns=true_matrix.columns, fill_value=0)
    mask = true_matrix.values > 0
    mse = mean_squared_error(true_matrix.values[mask], pred_matrix.values[mask])
    mae = mean_absolute_error(true_matrix.values[mask], pred_matrix.values[mask])
    return np.sqrt(mse), mae

svd_rmse, svd_mae = compute_rmse_mae(approx_matrix, test_matrix)
ibcf_rmse, ibcf_mae = compute_rmse_mae(ibcf_pred_matrix, test_matrix)

print(f"SVD RMSE: {svd_rmse:.4f}, MAE: {svd_mae:.4f}")
print(f"Item-Based CF RMSE: {ibcf_rmse:.4f}, MAE: {ibcf_mae:.4f}")

# 7️⃣ TOP-5 rekomendacje
def top_n(pred_matrix, user_id, n=5):
    user_pred = pred_matrix.loc[user_id]
    top_items = user_pred.sort_values(ascending=False).index[:n].tolist()
    return top_items

example_user = train_matrix.index[3]
print("SVD TOP-5:", top_n(approx_matrix, example_user))
print("Item-Based CF TOP-5:", top_n(ibcf_pred_matrix, example_user))

# 8️⃣ Wybór najlepszego modelu
if svd_rmse < ibcf_rmse:
    best_model = ('svd', svd)
else:
    best_model = ('ibcf', item_similarity)

# 9️⃣ Zapis modelu
os.makedirs('best_model', exist_ok=True)
model_path = os.path.join('best_model', 'best_model.pkl')
with open(model_path, 'wb') as f:
    pickle.dump(best_model, f)

print(f"Najlepszy model: {best_model[0]} zapisany w {model_path}")


SVD RMSE: 2.9445, MAE: 2.7100
Item-Based CF RMSE: 7.8099, MAE: 5.0564
SVD TOP-5: [1197.0, 1196.0, 1210.0, 2716.0, 260.0]
Item-Based CF TOP-5: [588.0, 590.0, 440.0, 442.0, 145.0]
Najlepszy model: svd zapisany w best_model\best_model.pkl
